# 🏡 House Price Prediction - Production Pipeline

## 🎯 Objective
This notebook demonstrates an **industry-ready end-to-end Machine Learning pipeline** for predicting residential property prices. 

### Key Features:
- **Custom Synthetic Dataset:** 10,000 perfectly balanced records ensuring no bias across property types.
- **Advanced Preprocessing:** Uses a `Scikit-Learn Pipeline` with `OrdinalEncoder` to explicitly preserve the logical hierarchy of property classes.
- **Predictive Modeling:** Implements an **XGBoost Regressor** optimized for high accuracy and market-realistic variance.
- **Deployment Ready:** Automatically saves a serialized pipeline artifact for use in real-time applications (Flask).

### 📥 Step 1: Dependencies & Data Loading
We use standard data science libraries and load our strictly-defined synthetic dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
df = pd.read_csv('Synthetic4_local_housePrice.csv')

# Initial look at the data
print(f"Dataset Shape: {df.shape}")
df.head()

### 🧹 Step 2: Data Cleaning & Normalization
Ensuring consistency in categorical values and numeric formatting.

In [ ]:
# Normalize text cases to prevent duplicate categories
df['Status'] = df['Status'].str.strip().str.capitalize()
df['Type'] = df['Type'].str.strip().str.capitalize()

# Handle price formatting if necessary
if df['Price'].dtype == 'O':
    df['Price'] = df['Price'].str.replace(',', '').astype(float)

print("Unique Property Types:", df['Type'].unique())
print("Unique Construction Statuses:", df['Status'].unique())

### 📐 Step 3: Feature Engineering
We predict **Price per Sq.Ft.** instead of Total Price. This avoids mathematical bias against smaller properties and ensures the model learns the underlying market rate.

In [ ]:
# Target variable: Rate per Square Foot
df['Price_per_sqft'] = df['Price'] / df['Area']

X = df[['BHK', 'Type', 'Area', 'Status']]
y = df['Price_per_sqft']

# Split into Training and Testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

### 🏗️ Step 4: Building the Pipeline
**What**: Creating a `ColumnTransformer` to handle both numeric and categorical data.
**Why**: Machine learning models like XGBoost require numeric input. We use `OrdinalEncoder` to preserve the logical hierarchy of property types (Apartment < Row bunglow < Independent house).
**How**: Using `sklearn.compose.ColumnTransformer` and `sklearn.preprocessing.OrdinalEncoder`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

# Define explicit categories for Ordinal Encoding to preserve hierarchy
type_order = ['Apartment', 'Row bunglow', 'Independent house']
status_order = ['Under construction', 'Ready to move']

# Create preprocessor with OrdinalEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=[type_order, status_order]), ['Type', 'Status'])
    ],
    remainder='passthrough' # Keep BHK and Area as is
)

### 🚀 Step 5: Model Training and Evaluation
**What**: Training an **XGBoost Regressor** using our optimized pipeline.
**Why**: XGBoost (Extreme Gradient Boosting) is an industry-leading algorithm known for its high accuracy and efficiency in regression tasks.
**How**: Integrating `xgboost.XGBRegressor` into our Scikit-Learn Pipeline.

In [ ]:
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Build and train XGBoost Pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42))
])

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'XGBoost R2 Score: {r2:.4f}')
print(f'XGBoost MAE: {mae:.2f} per sq.ft.')

### 💾 Step 6: Artifact Serialization
Saving the trained pipeline so it can be loaded by the Streamlit application.

In [ ]:
if not os.path.exists('artifacts'):
    os.makedirs('artifacts')

joblib.dump(model_pipeline, 'artifacts/house_price_model.pkl')
print('✅ XGBoost Pipeline saved successfully to artifacts/house_price_model.pkl')